In [ ]:
# ── Clone repo from GitHub (run once per Colab session) ───────────────────
import os, sys
if not os.path.exists('/content/protosearch'):
    !git clone https://github.com/bjdraper/protosearch.git /content/protosearch
sys.path.insert(0, '/content/protosearch')
!pip install -q -e /content/protosearch

# 04 — Structures
1. Download AlphaFold v6 structures for human reference proteins
2. Select centroid sequence per cluster
3. Fold centroids with local ESMFold
4. Compare structures with py3Dmol + RMSD table

In [ ]:
import os, sys
from pathlib import Path

# ── CONFIG (edit here) ────────────────────────────────────────────────────────
PROJECT_ROOT  = '/content/drive/MyDrive/acyltransferase'

# Which clusters to select centroids for (must match cluster label in assignments TSV)
CLUSTERS = {
    'lclat1':        'LCLAT1',
    'gpat4':         'GPAT4',
    'tafazzin_ctrl': 'Tafazzin [ctrl]',
    'lpcat2':        'LPCAT2',
    'gpat3':         'GPAT3',
}
# Cluster centroid colours for overlay view
CLUSTER_COLOURS = {
    'lclat1':        '#FF8F00',
    'gpat4':         '#00897B',
    'tafazzin_ctrl': '#8E24AA',
    'lpcat2':        '#1E88E5',
    'gpat3':         '#43A047',
}
# ─────────────────────────────────────────────────────────────────────────────

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from acyltransferase.config import load_config
cfg = load_config('config.yaml')

In [ ]:
# 1. Download AlphaFold structures for human references
from acyltransferase.structure import download_alphafold_batch

af_dir = Path('results/structures/alphafold')
af_pdbs = download_alphafold_batch(cfg.alphafold_targets, af_dir)

ok = sum(1 for v in af_pdbs.values() if v)
print(f'{ok}/{len(cfg.alphafold_targets)} AlphaFold structures downloaded')

In [ ]:
# 2. Select centroid sequence per cluster
import numpy as np
import pandas as pd
from acyltransferase.embed     import load_embeddings
from acyltransferase.structure import select_centroid
from acyltransferase.utils     import write_fasta

embeddings, embed_ids = load_embeddings(
    ('data/embeddings/hits.npy', 'data/embeddings/hits_ids.txt'),
)
assignments = pd.read_csv('results/clustering/cluster_assignments.tsv', sep='\t', dtype=str)
hits_faa    = 'data/hmmer_hits/proteins_filtered_hmmer.faa'

centroids = {}
for slug, label in CLUSTERS.items():
    result = select_centroid(
        cluster_label  = label,
        assignments_df = assignments,
        embeddings     = embeddings,
        embed_ids      = embed_ids,
        source_fastas  = [hits_faa],
    )
    if result:
        pid, seq = result
        centroids[slug] = (pid, seq)
        print(f'  {slug}: centroid = {pid}  ({len(seq)} aa)')
    else:
        print(f'  {slug}: not found')

write_fasta(
    [(f'{slug}_centroid', seq) for slug, (pid, seq) in centroids.items()],
    'results/structures/centroids.faa',
)

In [ ]:
# 3. Fold centroids with ESMFold (GPU recommended — ~3 min/seq on T4)
from acyltransferase.structure import fold_esmfold

esm_dir  = Path('results/structures/esmfold')
esm_dir.mkdir(parents=True, exist_ok=True)
esm_pdbs = {}

for slug, (pid, seq) in centroids.items():
    pdb = fold_esmfold(seq, label=slug, output_dir=esm_dir)
    esm_pdbs[slug] = pdb

print('\nESMFold structures:')
for slug, pdb in esm_pdbs.items():
    print(f'  {slug}: {pdb}')

In [ ]:
# 4a. View a single structure
from acyltransferase.structure_viz import view_single

# Change 'lclat1' to any cluster name from CLUSTERS
view = view_single(
    pdb_path    = esm_pdbs['lclat1'],
    colour      = 'spectrum',
    style       = 'cartoon',
    motif_config = cfg.catalytic_motifs,
)
view.show()  # renders interactively in Colab

In [ ]:
# 4b. Overlay all centroid structures
from acyltransferase.structure_viz import view_overlay

pdb_paths = {slug: esm_pdbs[slug] for slug in CLUSTER_COLOURS if esm_pdbs.get(slug)}

view = view_overlay(
    pdb_paths   = pdb_paths,
    colours     = CLUSTER_COLOURS,
    motif_config = cfg.catalytic_motifs,
    transparency = 0.0,
    width=1000, height=700,
)
view.show()

In [ ]:
# 4c. RMSD table (C-alpha, no structural alignment)
from acyltransferase.structure_viz import rmsd_table
import pandas as pd

rmsd_df = rmsd_table(pdb_paths)
rmsd_df.to_csv('results/structures/rmsd_table.tsv', sep='\t', index=False)

# Pivot to matrix view
labels = list(pdb_paths.keys())
import numpy as np
mat = np.zeros((len(labels), len(labels)))
for _, row in rmsd_df.iterrows():
    i, j = labels.index(row['structure_A']), labels.index(row['structure_B'])
    mat[i, j] = mat[j, i] = row['RMSD_Angstrom']

import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(mat, xticklabels=labels, yticklabels=labels,
            annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('Pairwise C-alpha RMSD (Å) — centroids')
plt.tight_layout()
plt.savefig('results/structures/rmsd_heatmap.png', dpi=150)
plt.show()

In [ ]:
# 4d. Compare one centroid vs AlphaFold human reference
COMPARE_CLUSTER = 'lclat1'
AF_REF_PATH     = list(Path('results/structures/alphafold').glob('AF-Q6UWP7-F1-model_v*.pdb'))

if AF_REF_PATH and esm_pdbs.get(COMPARE_CLUSTER):
    view = view_overlay(
        pdb_paths = {
            f'{COMPARE_CLUSTER}_ESMFold': esm_pdbs[COMPARE_CLUSTER],
            'LCLAT1_AlphaFold':           AF_REF_PATH[0],
        },
        colours      = {f'{COMPARE_CLUSTER}_ESMFold': '#FF8F00', 'LCLAT1_AlphaFold': '#AAAAAA'},
        transparency = 0.3,
        show_motifs  = True,
    )
    view.show()

## ✓ Done
All 4 notebooks complete. Results saved to `results/`.